In [ ]:
# ===== 5-Fold CV: ResNet50 / ResNet18 / MobileNetV3-L / EfficientNet-B0 / ViT-B16 / Swin-T =====
import os, json, random, time, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

# ---------------- CONFIG ----------------
DATA_DIR   = "/kaggle/input/datasets/sabbir4724/dentiro1"
OUT_DIR    = "/kaggle/working/kfold_results"
EPOCHS     = 40
N_FOLDS    = 5
IMG_SIZE   = 224
BATCH_SIZE = 32
LR         = 1e-4
WD         = 1e-2
SEED       = 42
NUM_WORKERS= 0
PRETRAINED = True
DPI        = 600
MODELS     = ["resnet50", "resnet18", "mobilenetv3_large", "efficientnet_b0", "vit_b_16", "swin_tiny"]
# ----------------------------------------

CKPT_DIR = f"{OUT_DIR}/ckpt"; RES_DIR = f"{OUT_DIR}/folds"; PLOT_DIR = f"{OUT_DIR}/plots"; CSV_DIR = f"{OUT_DIR}/csv"
for d in [CKPT_DIR, RES_DIR, PLOT_DIR, CSV_DIR]: os.makedirs(d, exist_ok=True)

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------------- DATA ----------------
base = datasets.ImageFolder(DATA_DIR)
samples = base.samples
paths   = np.array([s[0] for s in samples])
labels  = np.array([s[1] for s in samples])
CLASSES = base.classes
NUM_CLASSES = len(CLASSES)
print(f"Images: {len(paths)} | Classes ({NUM_CLASSES}): {CLASSES}")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class DS(Dataset):
    def __init__(self, p, l, tf): self.p, self.l, self.tf = p, l, tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        return self.tf(Image.open(self.p[i]).convert("RGB")), int(self.l[i])

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
SPLITS = list(skf.split(paths, labels))

# ---------------- MODELS ----------------
def build_model(name):
    w = "DEFAULT" if PRETRAINED else None
    if name == "resnet50":
        m = models.resnet50(weights=w); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "resnet18":
        m = models.resnet18(weights=w); m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    elif name == "mobilenetv3_large":
        m = models.mobilenet_v3_large(weights=w); m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
    elif name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=w); m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
    elif name == "vit_b_16":
        m = models.vit_b_16(weights=w); m.heads.head = nn.Linear(m.heads.head.in_features, NUM_CLASSES)
    elif name == "swin_tiny":
        m = models.swin_t(weights=w); m.head = nn.Linear(m.head.in_features, NUM_CLASSES)
    else:
        raise ValueError(name)
    return m.to(device)

# ---------------- TRAIN / EVAL ----------------
@torch.no_grad()
def evaluate(model, loader, crit):
    model.eval(); tl, yt, yp = 0.0, [], []
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with torch.autocast("cuda", enabled=device.type == "cuda"):
            o = model(x); loss = crit(o, y)
        tl += loss.item() * x.size(0)
        yt += y.cpu().tolist(); yp += o.argmax(1).cpu().tolist()
    return tl / len(yt), accuracy_score(yt, yp), f1_score(yt, yp, average="macro"), np.array(yt), np.array(yp)

def run_fold(name, fold):
    res_path = f"{RES_DIR}/{name}_fold{fold}.json"
    if os.path.exists(res_path):
        return json.load(open(res_path))
    tr_idx, va_idx = SPLITS[fold]
    tr_ld = DataLoader(DS(paths[tr_idx], labels[tr_idx], train_tf), BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_ld = DataLoader(DS(paths[va_idx], labels[va_idx], val_tf), BATCH_SIZE * 2, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True)
    seed_all(SEED + fold)
    model = build_model(name)
    crit  = nn.CrossEntropyLoss()
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_f1"]}
    best = {"f1": -1, "epoch": -1, "report": None, "cm": None}
    start = 0
    ck_path = f"{CKPT_DIR}/{name}_fold{fold}_last.pth"
    if os.path.exists(ck_path):   # resume after kernel reset
        ck = torch.load(ck_path, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        hist, best, start = ck["hist"], ck["best"], ck["epoch"] + 1
        print(f"[RESUME] {name} fold{fold} from epoch {start}")

    for ep in range(start, EPOCHS):
        model.train(); tl, correct, n = 0.0, 0, 0
        for x, y in tqdm(tr_ld, desc=f"{name} F{fold} E{ep+1}/{EPOCHS}", leave=False):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast("cuda", enabled=device.type == "cuda"):
                o = model(x); loss = crit(o, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * x.size(0); correct += (o.argmax(1) == y).sum().item(); n += x.size(0)
        sched.step()
        vl, va, vf, yt, yp = evaluate(model, va_ld, crit)
        for k, v in zip(hist.keys(), [tl / n, correct / n, vl, va, vf]): hist[k].append(v)
        if vf > best["f1"]:
            rep = classification_report(yt, yp, target_names=CLASSES, output_dict=True, zero_division=0)
            best = {"f1": vf, "epoch": ep + 1, "report": rep, "cm": confusion_matrix(yt, yp).tolist()}
        print(f"{name} F{fold} E{ep+1:02d} | trL {tl/n:.4f} trA {correct/n:.4f} | vaL {vl:.4f} vaA {va:.4f} vaF1 {vf:.4f}")
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "hist": hist, "best": best, "epoch": ep}, ck_path)

    out = {"hist": hist, "best": best}
    json.dump(out, open(res_path, "w"))
    if os.path.exists(ck_path): os.remove(ck_path)   # disk save
    del model, opt; torch.cuda.empty_cache()
    return out

# ---------------- REPORTS & PLOTS ----------------
def mean_std_plot(ax, arr, color, label=None):
    m, s = arr.mean(0), arr.std(0); x = np.arange(1, arr.shape[1] + 1)
    ax.plot(x, m, color=color, lw=2, label=label); ax.fill_between(x, m - s, m + s, color=color, alpha=0.25)

def report_model(name, results):
    cols = plt.cm.tab10(np.arange(N_FOLDS))
    H = {k: np.array([r["hist"][k] for r in results]) for k in results[0]["hist"]}
    ep = np.arange(1, EPOCHS + 1)

    # --- individual plots + combined 2x3 ---
    specs = [("train_loss", "Training Loss (all folds)", "Loss", "all"),
             ("val_acc", "Validation Accuracy (all folds)", "Accuracy", "all"),
             ("val_f1", "Validation Macro-F1 (all folds)", "Macro-F1", "all"),
             ("train_loss", "Mean Training Loss ± Std", "Loss", "ms"),
             ("val_acc", "Mean Validation Accuracy ± Std", "Accuracy", "ms"),
             ("val_f1", "Mean Validation Macro-F1 ± Std", "Macro-F1", "ms")]
    def draw(ax, key, title, yl, mode):
        if mode == "all":
            for f in range(N_FOLDS): ax.plot(ep, H[key][f], color=cols[f], lw=1.5, label=f"Fold {f}")
            ax.legend(fontsize=8)
        else:
            mean_std_plot(ax, H[key], "tab:blue", "mean ± std"); ax.legend(fontsize=8)
        ax.set_title(f"{name}: {title}", fontsize=10); ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
    fig, axs = plt.subplots(2, 3, figsize=(18, 9))
    for ax, sp in zip(axs.ravel(), specs): draw(ax, *sp)
    plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/{name}_combined.png", dpi=DPI, bbox_inches="tight"); plt.show()
    for i, sp in enumerate(specs):
        fig, ax = plt.subplots(figsize=(6, 4.5)); draw(ax, *sp)
        plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/{name}_{i+1}_{sp[0]}_{sp[3]}.png", dpi=DPI, bbox_inches="tight"); plt.close()

    # --- fold-wise per-class results (best epoch by val macro-F1) ---
    rows = []
    for f, r in enumerate(results):
        rep = r["best"]["report"]
        for c in CLASSES:
            rows.append({"model": name, "fold": f, "best_epoch": r["best"]["epoch"], "class": c,
                         "precision": rep[c]["precision"], "recall": rep[c]["recall"],
                         "f1": rep[c]["f1-score"], "support": rep[c]["support"]})
    df = pd.DataFrame(rows); df.to_csv(f"{CSV_DIR}/{name}_perclass_foldwise.csv", index=False)
    print(f"\n===== {name}: fold-wise per-class results =====")
    for f in range(N_FOLDS):
        print(f"\n-- Fold {f} (best epoch {results[f]['best']['epoch']}) --")
        print(df[df.fold == f][["class", "precision", "recall", "f1", "support"]].round(4).to_string(index=False))
    pc = df.groupby("class")[["precision", "recall", "f1"]].agg(["mean", "std"]).round(4)
    pc.to_csv(f"{CSV_DIR}/{name}_perclass_mean_std.csv")
    print(f"\n-- {name}: per-class mean ± std over folds --"); print(pc)

    # --- fold summary ---
    summ = pd.DataFrame([{"model": name, "fold": f, "best_epoch": r["best"]["epoch"],
                          "val_acc": r["hist"]["val_acc"][r["best"]["epoch"] - 1],
                          "val_macro_f1": r["best"]["f1"]} for f, r in enumerate(results)])
    summ.to_csv(f"{CSV_DIR}/{name}_fold_summary.csv", index=False)
    print(f"\n-- {name}: fold summary --"); print(summ.round(4).to_string(index=False))
    print(f"Mean Acc {summ.val_acc.mean():.4f} ± {summ.val_acc.std():.4f} | "
          f"Mean Macro-F1 {summ.val_macro_f1.mean():.4f} ± {summ.val_macro_f1.std():.4f}")
    return H, summ

# ---------------- MAIN ----------------
ALL_H, ALL_SUMM = {}, []
for name in MODELS:
    t0 = time.time(); results = []
    for fold in range(N_FOLDS):
        results.append(run_fold(name, fold))
    H, summ = report_model(name, results)
    ALL_H[name] = H; ALL_SUMM.append(summ)
    print(f"[DONE] {name} in {(time.time()-t0)/60:.1f} min")

# ---------------- cross-model comparison ----------------
final = pd.concat(ALL_SUMM)
tab = final.groupby("model").agg(acc_mean=("val_acc", "mean"), acc_std=("val_acc", "std"),
                                 f1_mean=("val_macro_f1", "mean"), f1_std=("val_macro_f1", "std")).round(4)
tab.to_csv(f"{CSV_DIR}/all_models_summary.csv"); print("\n===== ALL MODELS =====\n", tab)

pal = plt.cm.tab10(np.arange(len(MODELS)))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (key, t) in zip(axs, [("train_loss", "Mean Training Loss ± Std"),
                              ("val_acc", "Mean Validation Accuracy ± Std"),
                              ("val_f1", "Mean Validation Macro-F1 ± Std")]):
    for i, n in enumerate(ALL_H): mean_std_plot(ax, ALL_H[n][key], pal[i], n)
    ax.set_title(t); ax.set_xlabel("Epoch"); ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/all_models_mean_std.png", dpi=DPI, bbox_inches="tight"); plt.show()

In [ ]:
# ===== Summary figure: all folds, all models =====
import os, json
import numpy as np
import matplotlib.pyplot as plt

OUT_DIR  = "/kaggle/working/kfold_results"
RES_DIR  = f"{OUT_DIR}/folds"
PLOT_DIR = f"{OUT_DIR}/plots"
os.makedirs(PLOT_DIR, exist_ok=True)

MODELS  = ["resnet50", "resnet18", "mobilenetv3_large", "efficientnet_b0", "vit_b_16"]
N_FOLDS = 5
DPI     = 1000

METRICS = [("train_loss", "Training Loss",            "Loss"),
           ("val_acc",    "Validation Accuracy",      "Accuracy"),
           ("val_f1",     "Validation Macro-F1",      "Macro-F1")]

# ---- load histories ----
H = {}
for m in MODELS:
    runs = [json.load(open(f"{RES_DIR}/{m}_fold{f}.json"))["hist"] for f in range(N_FOLDS)]
    H[m] = {k: np.array([r[k] for r in runs]) for k in runs[0]}   # (folds, epochs)

cols = plt.cm.tab10(np.arange(N_FOLDS))

# ---- Figure 1: all folds, all models ----
fig, axs = plt.subplots(len(MODELS), 3, figsize=(18, 3.6 * len(MODELS)))
for i, m in enumerate(MODELS):
    for j, (key, title, yl) in enumerate(METRICS):
        ax = axs[i, j]; arr = H[m][key]; ep = np.arange(1, arr.shape[1] + 1)
        for f in range(N_FOLDS):
            ax.plot(ep, arr[f], color=cols[f], lw=1.4, label=f"Fold {f}")
        ax.set_title(f"{m}: {title}", fontsize=10)
        ax.set_xlabel("Epoch"); ax.set_ylabel(yl); ax.grid(alpha=0.3)
        if i == 0 and j == 0: ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/summary_allfolds_all_models.png", dpi=DPI, bbox_inches="tight")
plt.show()

# ---- Figure 2: model-wise mean ± std (3 panel, all models in one plot) ----
pal = plt.cm.tab10(np.arange(len(MODELS)))
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
for ax, (key, title, yl) in zip(axs, METRICS):
    for i, m in enumerate(MODELS):
        arr = H[m][key]; mu, sd = arr.mean(0), arr.std(0); ep = np.arange(1, arr.shape[1] + 1)
        ax.plot(ep, mu, color=pal[i], lw=2, label=m)
        ax.fill_between(ep, mu - sd, mu + sd, color=pal[i], alpha=0.2)
    ax.set_title(f"Mean {title} ± Std"); ax.set_xlabel("Epoch"); ax.set_ylabel(yl)
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{PLOT_DIR}/summary_mean_std_all_models.png", dpi=DPI, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Proposed vs Baselines comparison =====
import os, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
from torchvision import models
from scipy import stats

OUT_DIR  = "/kaggle/working/kfold_results"
PLOT_DIR = f"{OUT_DIR}/plots"; CSV_DIR = f"{OUT_DIR}/csv"
os.makedirs(PLOT_DIR, exist_ok=True); os.makedirs(CSV_DIR, exist_ok=True)
BASE    = ["resnet50", "resnet18", "mobilenetv3_large", "efficientnet_b0", "vit_b_16"]
N_FOLDS = 5; DPI = 600
NC = 5

# ---------- Proposed model ----------
PROP_NAME   = "DentAttn-MSFNet"
PROP_PARAMS = 1.684
P_ACC = [0.993396, 0.974528, 0.985836, 0.982075, 0.986805]
P_F1  = [0.989890, 0.961193, 0.981125, 0.972641, 0.981118]
P_WF1 = [0.993419, 0.974270, 0.985800, 0.981936, 0.986806]

# ---------- Baselines  ----------
M = {
    "resnet50":          {"acc": [0.9672, 0.9719, 0.9707, 0.9766, 0.9637],
                          "f1":  [0.9283, 0.9426, 0.9361, 0.9457, 0.9124]},
    "resnet18":          {"acc": [0.9555, 0.9707, 0.9613, 0.9695, 0.9613],
                          "f1":  [0.9020, 0.9332, 0.9275, 0.9366, 0.9114]},
    "mobilenetv3_large": {"acc": [0.9567, 0.9684, 0.9660, 0.9648, 0.9637],
                          "f1":  [0.9022, 0.9292, 0.9194, 0.9145, 0.9195]},
    "efficientnet_b0":   {"acc": [0.9649, 0.9766, 0.9683, 0.9637, 0.9637],
                          "f1":  [0.9220, 0.9502, 0.9380, 0.9317, 0.9156]},
    "vit_b_16":          {"acc": [0.9625, 0.9625, 0.9707, 0.9566, 0.9590],
                          "f1":  [0.9156, 0.9174, 0.9449, 0.9018, 0.9223]},
}
M[PROP_NAME] = {"acc": P_ACC, "f1": P_F1, "wf1": P_WF1}

# Weighted-F1 baseline
WF1_STR = {"resnet50": "0.9695 ± 0.0049", "resnet18": "0.9631 ± 0.0063",
           "mobilenetv3_large": "0.9635 ± 0.0045", "efficientnet_b0": "0.9672 ± 0.0057",
           "vit_b_16": "0.9615 ± 0.0058"}

# Backup:
json.dump(M, open(f"{OUT_DIR}/fold_metrics_backup.json", "w"), indent=2)

# ---------- Params  ----------
def build(name):
    if name == "resnet50": n = models.resnet50(weights=None); n.fc = nn.Linear(n.fc.in_features, NC)
    elif name == "resnet18": n = models.resnet18(weights=None); n.fc = nn.Linear(n.fc.in_features, NC)
    elif name == "mobilenetv3_large": n = models.mobilenet_v3_large(weights=None); n.classifier[3] = nn.Linear(n.classifier[3].in_features, NC)
    elif name == "efficientnet_b0": n = models.efficientnet_b0(weights=None); n.classifier[1] = nn.Linear(n.classifier[1].in_features, NC)
    elif name == "vit_b_16": n = models.vit_b_16(weights=None); n.heads.head = nn.Linear(n.heads.head.in_features, NC)
    return n
PARAMS = {m: sum(p.numel() for p in build(m).parameters()) / 1e6 for m in BASE}
PARAMS[PROP_NAME] = PROP_PARAMS

# ---------- Multiple-comparison helpers ----------
def holm(p):
    p = np.asarray(p, float); m = len(p); order = np.argsort(p)
    adj = np.empty(m); run = 0.0
    for rank, idx in enumerate(order):
        run = max(run, (m - rank) * p[idx]); adj[idx] = min(1.0, run)
    return adj
def bonf(p): return np.minimum(1.0, np.asarray(p, float) * len(p))

# ---------- 1) Comparison table ----------
ms = lambda v: f"{np.mean(v):.4f} ± {np.std(v, ddof=1):.4f}"
rows = []
for m in BASE + [PROP_NAME]:
    wf1 = ms(M[m]["wf1"]) if m == PROP_NAME else WF1_STR[m]
    rows.append({"model": m, "params(M)": round(PARAMS[m], 3),
                 "accuracy": ms(M[m]["acc"]), "macro_f1": ms(M[m]["f1"]), "weighted_f1": wf1,
                 "mean_f1": np.mean(M[m]["f1"]),
                 "F1 per M-param": round(np.mean(M[m]["f1"]) / PARAMS[m], 4)})
tab = pd.DataFrame(rows).sort_values("mean_f1", ascending=False).drop(columns="mean_f1")
tab.to_csv(f"{CSV_DIR}/proposed_vs_baselines.csv", index=False)
print("===== PROPOSED vs BASELINES =====\n", tab.to_string(index=False))

# ---------- 2) Delta + paired t-test + Holm/Bonferroni ----------
rows = []
for m in BASE:
    _, p_f1  = stats.ttest_rel(M[PROP_NAME]["f1"],  M[m]["f1"])
    _, p_acc = stats.ttest_rel(M[PROP_NAME]["acc"], M[m]["acc"])
    diff = np.array(M[PROP_NAME]["f1"]) - np.array(M[m]["f1"])
    se = diff.std(ddof=1) / np.sqrt(N_FOLDS)
    ci = stats.t.interval(0.95, N_FOLDS - 1, loc=diff.mean(), scale=se)
    rows.append({"baseline": m,
                 "param ratio (base/prop)": round(PARAMS[m] / PROP_PARAMS, 1),
                 "dAcc": np.mean(M[PROP_NAME]["acc"]) - np.mean(M[m]["acc"]),
                 "dF1": diff.mean(),
                 "dF1 95% CI": f"[{ci[0]:.4f}, {ci[1]:.4f}]",
                 "p_F1": p_f1, "p_Acc": p_acc,
                 "folds wins (F1)": f"{int((diff > 0).sum())}/{N_FOLDS}"})
d = pd.DataFrame(rows)
d["p_adj_Holm_F1"]  = holm(d["p_F1"]);   d["p_adj_Holm_Acc"]  = holm(d["p_Acc"])
d["p_adj_Bonf_F1"]  = bonf(d["p_F1"]);   d["p_adj_Bonf_Acc"]  = bonf(d["p_Acc"])
num = [c for c in d.columns if c.startswith(("p_", "d")) and c != "dF1 95% CI"]
d[num] = d[num].round(4)
d.to_csv(f"{CSV_DIR}/proposed_delta_tests.csv", index=False)
try:
    open(f"{CSV_DIR}/proposed_delta_tests.tex", "w").write(d.to_latex(index=False, float_format="%.4f"))
except Exception as e:
    print("LaTeX export skip:", e)
print("\n===== DELTA + PAIRED TEST (raw / Holm / Bonferroni) =====\n", d.to_string(index=False))

# ---------- 3) Fold-wise macro-F1 table ----------
fw = pd.DataFrame({m: M[m]["f1"] for m in BASE + [PROP_NAME]}, index=[f"Fold {i}" for i in range(N_FOLDS)])
fw.loc["Mean"] = fw.mean(); fw.loc["Std"] = fw.iloc[:N_FOLDS].std(ddof=1)
fw.round(4).to_csv(f"{CSV_DIR}/foldwise_f1_all.csv")
print("\n===== FOLD-WISE MACRO-F1 =====\n", fw.round(4))

# ---------- 4) Figures ----------
names = BASE + [PROP_NAME]
cols  = ["tab:gray"] * len(BASE) + ["tab:red"]

fig, axs = plt.subplots(1, 3, figsize=(20, 5.5))
mu = [np.mean(M[m]["f1"]) for m in names]; sd = [np.std(M[m]["f1"], ddof=1) for m in names]
axs[0].bar(names, mu, yerr=sd, color=cols, capsize=4)
axs[0].set_ylim(max(0, min(mu) - 0.05), 1.0); axs[0].set_title("Macro-F1 (mean ± std, 5-fold)")
axs[0].tick_params(axis="x", rotation=30); axs[0].grid(alpha=0.3, axis="y")
for i, v in enumerate(mu): axs[0].text(i, v + sd[i] + 0.002, f"{v:.3f}", ha="center", fontsize=8)

for m, c in zip(names, cols):
    axs[1].scatter(PARAMS[m], np.mean(M[m]["f1"]), s=140 if m == PROP_NAME else 80, color=c)
    axs[1].annotate(m, (PARAMS[m], np.mean(M[m]["f1"])), textcoords="offset points", xytext=(6, 6), fontsize=9)
axs[1].set_xscale("log"); axs[1].set_xlabel("Parameters (M, log scale)"); axs[1].set_ylabel("Macro-F1")
axs[1].set_title("Accuracy–Efficiency trade-off"); axs[1].grid(alpha=0.3)

bp = axs[2].boxplot([M[m]["f1"] for m in names], showmeans=True, patch_artist=True)
axs[2].set_xticklabels(names)
for patch, c in zip(bp["boxes"], cols): patch.set_facecolor(c); patch.set_alpha(0.5)
axs[2].set_title("Fold-wise Macro-F1"); axs[2].tick_params(axis="x", rotation=30); axs[2].grid(alpha=0.3)
plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/proposed_vs_baselines.png", dpi=DPI, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
for m, c in zip(names, cols):
    ax.plot(range(N_FOLDS), M[m]["f1"], marker="o", color=c, lw=2.5 if m == PROP_NAME else 1.3, label=m)
ax.set_xticks(range(N_FOLDS)); ax.set_xlabel("Fold"); ax.set_ylabel("Macro-F1")
ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_title("Per-fold Macro-F1")
plt.tight_layout(); plt.savefig(f"{PLOT_DIR}/proposed_vs_baselines_perfold.png", dpi=DPI, bbox_inches="tight"); plt.show()